# Using Online Customer Rating to Examine Restaurant Health Inspection Results

**Team members:** Raffaella Alpaca, Helen Ayalew, Angel Pavu

**Repository:** https://github.com/hwayalew/ds3001_team1_project

## Setup

Use relative paths and keep the notebook runnable from beginning to end.

In [18]:
from pathlib import Path

# Download our team's GitHub repository into Colab
!git clone https://github.com/hwayalew/ds3001_team1_project.git

# Set the path to our data folder
DATA_DIR = Path("ds3001_team1_project/data")

# Check that the folder exists
assert DATA_DIR.is_dir(), "The data directory is missing."
print("Data folder found!")


fatal: destination path 'ds3001_team1_project' already exists and is not an empty directory.
Data folder found!


In [19]:
# Show all files inside our data folder
for file in DATA_DIR.iterdir():
    print(file.name)


food_inspections_all.xlsx
fairfax_falls_church_google_restaurants.csv
README.md


## Load data

In [20]:
# Load data from DATA_DIR.

In [21]:
import pandas as pd
# Load the food inspection data from Excel
inspections = pd.read_excel(
    DATA_DIR / "food_inspections_all.xlsx",
    sheet_name="Inspections"
)
# Load the establishment data from the same Excel file
establishments = pd.read_excel(
    DATA_DIR / "food_inspections_all.xlsx",
    sheet_name="Establishments"
)

# Load the Google restaurant ratings from CSV
google = pd.read_csv(
    DATA_DIR / "fairfax_falls_church_google_restaurants.csv"
)

# Check that everything loaded correctly
print("Inspections:", inspections.shape)
print("Establishments:", establishments.shape)
print("Google Ratings:", google.shape)

Inspections: (13271, 13)
Establishments: (3840, 9)
Google Ratings: (876, 12)


## Prepare data

In [22]:
# Clean data and create features.

In [23]:
# Prepare data for analysis
# Make copies so we keep the original datasets unchanged
inspections_clean = inspections.copy()
establishments_clean = establishments.copy()
google_clean = google.copy()

#check missing values in each dataset
print("Missing values in inspections:")
print(inspections_clean.isnull().sum())

print("\nMissing values in establishments:")
print(establishments_clean.isnull().sum())

print("\nMissing values in Google ratings:")
print(google_clean.isnull().sum())

#check the column names before cleaning and merging
print("\nInspection columns:")
print(inspections_clean.columns.tolist())

print("\nEstablishment columns:")
print(establishments_clean.columns.tolist())

print("\nGoogle columns:")
print(google_clean.columns.tolist())


Missing values in inspections:
INSPECTIONID           0
RECORDID              11
ESTABLISHMENT_NAME     1
ESTABLISHMENT_TYPE    12
ADDRESS               12
CITY                  12
STATE                 12
ZIP                   12
JURISDICTION          12
FOOD_PLACE            12
INSPECTIONDATE         0
INSPECTIONRESULT       0
WEBSITE_LINK           0
dtype: int64

Missing values in establishments:
RECORDID              0
ESTABLISHMENT_NAME    0
ESTABLISHMENT_TYPE    0
ADDRESS               0
CITY                  0
STATE                 0
ZIP                   0
JURISDICTION          0
FOOD_PLACE            0
dtype: int64

Missing values in Google ratings:
gmap_id              0
name                 0
address              0
city                 0
postal_code          0
categories           0
avg_rating           0
num_of_reviews       0
price              238
latitude             0
longitude            0
google_maps_url      0
dtype: int64

Inspection columns:
['INSPECTIONID', 'RECO

In [24]:
# Remove inspection records missing important information
#need these fields to identify and match restaurants!!!!
inspections_clean = inspections_clean.dropna(
    subset=["RECORDID", "ESTABLISHMENT_NAME", "ADDRESS", "CITY"]
)

# Google price is optional, so only remove rows missing essential information for matching and analysis
google_clean = google_clean.dropna(
    subset=["name", "address", "city", "avg_rating"]
)
#how many rows remain
print("Inspections after cleaning:", inspections_clean.shape)
print("Establishments after cleaning:", establishments_clean.shape)
print("Google ratings after cleaning:", google_clean.shape)


Inspections after cleaning: (13259, 13)
Establishments after cleaning: (3840, 9)
Google ratings after cleaning: (876, 12)


We remove 12 inspection rows missing key identifying information, provided the 11 missing RECORDID values are among those same 12 records.

In [25]:
#keep only Fairfax and Falls Church restaurants
cities = ["FAIRFAX", "FALLS CHURCH"]

#filter inspection data
inspections_clean = inspections_clean[
    inspections_clean["CITY"].str.upper().isin(cities)
].copy()

#filter establishment data
establishments_clean = establishments_clean[
    establishments_clean["CITY"].str.upper().isin(cities)
].copy()

#filter Google ratings
google_clean = google_clean[
    google_clean["city"].str.upper().isin(cities)
].copy()

#check the remaining rows
print("Inspections:", inspections_clean.shape)
print("Establishments:", establishments_clean.shape)
print("Google ratings:", google_clean.shape)


Inspections: (3863, 13)
Establishments: (1124, 9)
Google ratings: (876, 12)


In [26]:
#count each type of inspection result
print(inspections_clean["INSPECTIONRESULT"].value_counts())

INSPECTIONRESULT
Passed                  2958
Partial Pass             871
Failed - Self Closed      33
Failed                     1
Name: count, dtype: int64


In [27]:
#Check whether establishment IDs are unique
print("Duplicate establishment IDs:")
print(establishments_clean["RECORDID"].duplicated().sum())

# Check how many unique establishments have inspections
print("\nUnique establishments in inspections:")
print(inspections_clean["RECORDID"].nunique())

# Check whether inspection records have matching establishments
matches = inspections_clean["RECORDID"].isin(
    establishments_clean["RECORDID"]
)
print("\nInspections with matching establishments:")
print(matches.sum())

print("\nInspections without matching establishments:")
print((~matches).sum())


Duplicate establishment IDs:
0

Unique establishments in inspections:
1124

Inspections with matching establishments:
3863

Inspections without matching establishments:
0


In [28]:
#merge inspection records with establishment information
# Keep RECORDID from establishments for matching
establishment_ids = establishments_clean[["RECORDID"]]

# Combine the datasets using RECORDID
merged_inspections = pd.merge(
    inspections_clean,
    establishment_ids,
    on="RECORDID",
    how="inner",
    validate="many_to_one"
)

#check the merged dataset
print("Merged data shape:", merged_inspections.shape)
display(merged_inspections.head())

Merged data shape: (3863, 13)


,INSPECTIONID,RECORDID,ESTABLISHMENT_NAME,ESTABLISHMENT_TYPE,ADDRESS,CITY,STATE,ZIP,JURISDICTION,FOOD_PLACE,INSPECTIONDATE,INSPECTIONRESULT,WEBSITE_LINK
0,3534783,HFOOD-2023-00135,"""Robeks Fresh Juices & Smoothies""",Fast Food Restaurant,1063 BROAD ST,FALLS CHURCH,VA,22046,CITY OF FALLS CHURCH,Yes,2024-11-08,Passed,https://plus.fairfaxcounty.gov/CitizenAccess/u...
1,3677656,HFOOD-2023-00135,"""Robeks Fresh Juices & Smoothies""",Fast Food Restaurant,1063 BROAD ST,FALLS CHURCH,VA,22046,CITY OF FALLS CHURCH,Yes,2025-04-15,Passed,https://plus.fairfaxcounty.gov/CitizenAccess/u...
2,3877055,HFOOD-2023-00135,"""Robeks Fresh Juices & Smoothies""",Fast Food Restaurant,1063 BROAD ST,FALLS CHURCH,VA,22046,CITY OF FALLS CHURCH,Yes,2025-10-16,Passed,https://plus.fairfaxcounty.gov/CitizenAccess/u...
3,3529095,HFOOD-2021-00045,&PIZZA,Fast Food Restaurant,6110C ARLINGTON BLVD,FALLS CHURCH,VA,22044,FAIRFAX COUNTY,Yes,2024-11-01,Passed,https://plus.fairfaxcounty.gov/CitizenAccess/u...
4,3569188,HFOOD-2021-00310,&PIZZA,Fast Food Restaurant,11824KL FAIR OAKS SHOPPING CTR,FAIRFAX,VA,22033,FAIRFAX COUNTY,Yes,2024-12-16,Partial Pass,https://plus.fairfaxcounty.gov/CitizenAccess/u...


In [29]:
# Standardize restaurant names for matching
# Make copies to keep the original data unchanged
merged_inspections = merged_inspections.copy()
google_clean = google_clean.copy()

# Clean Fairfax restaurant names
merged_inspections["name_clean"] = (
    merged_inspections["ESTABLISHMENT_NAME"]
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.strip()
)
# Clean Google restaurant names
google_clean["name_clean"] = (
    google_clean["name"]
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.strip()
)
# Display examples
print("Fairfax restaurant names:")
display(
    merged_inspections[
        ["ESTABLISHMENT_NAME", "name_clean"]
    ].head()
)

print("Google restaurant names:")
display(
    google_clean[
        ["name", "name_clean"]
    ].head()
)

Fairfax restaurant names:


,ESTABLISHMENT_NAME,name_clean
0,"""Robeks Fresh Juices & Smoothies""",robeks fresh juices smoothies
1,"""Robeks Fresh Juices & Smoothies""",robeks fresh juices smoothies
2,"""Robeks Fresh Juices & Smoothies""",robeks fresh juices smoothies
3,&PIZZA,pizza
4,&PIZZA,pizza


Google restaurant names:


,name,name_clean
0,21 Great American Bistro,21 great american bistro
1,29 Diner,29 diner
2,703 Bar + Kitchen,703 bar kitchen
3,99C Hot Pot,99c hot pot
4,AABEE Persian & Mediterranean Restaurant,aabee persian mediterranean restaurant


In [30]:
# Standardize restaurant addresses for matching

# Clean Fairfax addresses
merged_inspections["address_clean"] = (
    merged_inspections["ADDRESS"]
    .str.lower()
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\bstreet\b", "st", regex=True)
    .str.replace(r"\broad\b", "rd", regex=True)
    .str.replace(r"\bavenue\b", "ave", regex=True)
    .str.replace(r"\bboulevard\b", "blvd", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Clean Google addresses
google_clean["address_clean"] = (
    google_clean["address"]
    .str.lower()
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\bstreet\b", "st", regex=True)
    .str.replace(r"\broad\b", "rd", regex=True)
    .str.replace(r"\bavenue\b", "ave", regex=True)
    .str.replace(r"\bboulevard\b", "blvd", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Preview the cleaned addresses
print("Fairfax addresses:")
display(merged_inspections[["ADDRESS", "address_clean"]].head())

print("Google addresses:")
display(google_clean[["address", "address_clean"]].head())

Fairfax addresses:


,ADDRESS,address_clean
0,1063 BROAD ST,1063 broad st
1,1063 BROAD ST,1063 broad st
2,1063 BROAD ST,1063 broad st
3,6110C ARLINGTON BLVD,6110c arlington blvd
4,11824KL FAIR OAKS SHOPPING CTR,11824kl fair oaks shopping ctr


Google addresses:


,address,address_clean
0,"21 Great American Bistro, 10427 North St #102,...",21 great american bistro 10427 north st 102 fa...
1,"29 Diner, 10536 Fairfax Blvd, Fairfax, VA 22030",29 diner 10536 fairfax blvd fairfax va 22030
2,"703 Bar + Kitchen, 11787 Lee Jackson Memorial ...",703 bar kitchen 11787 lee jackson memorial hwy...
3,"99C Hot Pot, 9992 Main St, Fairfax, VA 22031",99c hot pot 9992 main st fairfax va 22031
4,"AABEE Persian & Mediterranean Restaurant, Fair...",aabee persian mediterranean restaurant fairfax...


In [31]:
#extracts the street address from Google records

# Split the address into parts using commas
google_clean["street_address"] = (
    google_clean["address"]
    .str.split(",")
    .str[1]
    .str.strip()
)

# Clean the extracted street addresses
google_clean["street_address"] = (
    google_clean["street_address"]
    .str.lower()
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\bstreet\b", "st", regex=True)
    .str.replace(r"\broad\b", "rd", regex=True)
    .str.replace(r"\bavenue\b", "ave", regex=True)
    .str.replace(r"\bboulevard\b", "blvd", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Compare the original and extracted addresses
display(
    google_clean[
        ["name", "address", "street_address"]
    ].head(10)
)

# Check how many addresses could not be extracted
print("Missing street addresses:",
      google_clean["street_address"].isnull().sum())

,name,address,street_address
0,21 Great American Bistro,"21 Great American Bistro, 10427 North St #102,...",10427 north st 102
1,29 Diner,"29 Diner, 10536 Fairfax Blvd, Fairfax, VA 22030",10536 fairfax blvd
2,703 Bar + Kitchen,"703 Bar + Kitchen, 11787 Lee Jackson Memorial ...",11787 lee jackson memorial hwy
3,99C Hot Pot,"99C Hot Pot, 9992 Main St, Fairfax, VA 22031",9992 main st
4,AABEE Persian & Mediterranean Restaurant,"AABEE Persian & Mediterranean Restaurant, Fair...",fairfax blvd farr ave
5,ABC Canteen,"ABC Canteen, 9528 Lee Hwy, Fairfax, VA 22031",9528 lee hwy
6,Al-Maza Lebanese Mediterranean,"Al-Maza Lebanese Mediterranean, 10427 North St...",10427 north st
7,"Alta Strada, Mosaic","Alta Strada, Mosaic, 2911 District Ave #150, F...",mosaic
8,Angelico Pizza,"Angelico Pizza, 11391 Government Center Pkwy, ...",11391 government center pkwy
9,Angelika Film Center & Café at Mosaic,"Angelika Film Center & Café at Mosaic, 2911 Di...",2911 district ave suite 200


Missing street addresses: 0


not every extracted address is correct^^^

fixing the problem below:

In [32]:
import re

In [33]:
# Extract street addresses without confusing restaurant names

def get_street_address(row):
    # Get the complete address and restaurant name
    address = str(row["address"])
    name = str(row["name"])

    # Remove the restaurant name from the beginning
    if address.startswith(name + ","):
        address = address[len(name) + 1:].strip()

    # Split the remaining address into parts
    parts = address.split(",")

    # Find the first part that starts with a street number
    for part in parts:
        part = part.strip()

        if re.match(r"^\d+\s+", part):
            return part

    return None

# Apply the function to Google restaurants
google_clean["street_address"] = google_clean.apply(
    get_street_address,
    axis=1
)

# Standardize the street addresses
google_clean["street_address"] = (
    google_clean["street_address"]
    .str.lower()
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\bstreet\b", "st", regex=True)
    .str.replace(r"\broad\b", "rd", regex=True)
    .str.replace(r"\bavenue\b", "ave", regex=True)
    .str.replace(r"\bboulevard\b", "blvd", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Preview the corrected addresses
display(
    google_clean[
        ["name", "street_address"]
    ].head(10)
)

print("Missing street addresses:",
      google_clean["street_address"].isnull().sum())


,name,street_address
0,21 Great American Bistro,10427 north st 102
1,29 Diner,10536 fairfax blvd
2,703 Bar + Kitchen,11787 lee jackson memorial hwy
3,99C Hot Pot,9992 main st
4,AABEE Persian & Mediterranean Restaurant,None
5,ABC Canteen,9528 lee hwy
6,Al-Maza Lebanese Mediterranean,10427 north st
7,"Alta Strada, Mosaic",2911 district ave 150
8,Angelico Pizza,11391 government center pkwy
9,Angelika Film Center & Café at Mosaic,2911 district ave suite 200


Missing street addresses: 53


In [34]:
# Create one row per Fairfax establishment
fairfax_restaurants = merged_inspections.drop_duplicates(
    subset="RECORDID"
).copy()

# Standardize city names
fairfax_restaurants["city_clean"] = (
    fairfax_restaurants["CITY"].str.lower().str.strip()
)

google_clean["city_clean"] = (
    google_clean["city"].str.lower().str.strip()
)

# Find restaurants with the same cleaned name and city
possible_matches = pd.merge(
    fairfax_restaurants,
    google_clean,
    on=["name_clean", "city_clean"],
    how="inner",
    suffixes=("_fairfax", "_google")
)

# Count preliminary matches
print("Possible matching pairs:", len(possible_matches))

print("Unique Fairfax establishments matched:",
      possible_matches["RECORDID"].nunique())

print("Unique Google restaurants matched:",
      possible_matches["gmap_id"].nunique())

# Preview matches with both addresses
display(
    possible_matches[
        ["ESTABLISHMENT_NAME", "name",
         "ADDRESS", "street_address"]
    ].head(15)
)

Possible matching pairs: 350
Unique Fairfax establishments matched: 201
Unique Google restaurants matched: 262


,ESTABLISHMENT_NAME,name,ADDRESS,street_address
0,2941 RESTAURANT,2941 Restaurant,2941 FAIRVIEW PARK DR,2941 fairview park dr
1,ARBY'S,Arby's,8127 ROUTE 29,8127 lee hwy
2,ARIAKE JAPANESE RESTAURANT,Ariake Japanese Restaurant,8708 LITTLE RIVER TPKE,8708 little river turnpike
3,ARIGATO SUSHI,Arigato Sushi,11199 ROUTE 29,None
4,ARTIE'S,Artie's,3260 BLENHEIM BLVD,3260 old lee hwy
5,AUNTIE ANNE'S,Auntie Anne's,11953U FAIR OAKS SHOPPING CTR,11953 fair oaks mall
6,AUNTIE ANNE'S,Auntie Anne's,11761L FAIR OAKS SHOPPING CTR,11953 fair oaks mall
7,Anthony's Restaurant,Anthony's Restaurant,3000 ANNANDALE RD,3000 annandale rd
8,BABYLON FUTBOL CAFE,Babylon Futbol Cafe,3501A JEFFERSON ST,3501 s jefferson st
9,BAJA FRESH MEXICAN GRILL,Baja Fresh Mexican Grill,12150 FAIRFAX TOWNE CTR,None


In [35]:
#Compare street numbers

# Get the street number from Fairfax addresses
possible_matches["fairfax_number"] = (
    possible_matches["address_clean_fairfax"]
    .str.extract(r"^(\d+)")
)

# Get the street number from Google addresses
possible_matches["google_number"] = (
    possible_matches["street_address"]
    .str.extract(r"^(\d+)")
)

# Check whether street numbers match
possible_matches["number_match"] = (
    possible_matches["fairfax_number"]
    == possible_matches["google_number"]
)

# Count matching and non-matching addresses
print("Matching street numbers:",
      possible_matches["number_match"].sum())

print("Different or missing street numbers:",
      (~possible_matches["number_match"]).sum())

# Preview matching restaurants
display(
    possible_matches.loc[
        possible_matches["number_match"],
        ["ESTABLISHMENT_NAME", "name",
         "ADDRESS", "street_address"]
    ].head(15)
)


Matching street numbers: 169
Different or missing street numbers: 181


,ESTABLISHMENT_NAME,name,ADDRESS,street_address
0,2941 RESTAURANT,2941 Restaurant,2941 FAIRVIEW PARK DR,2941 fairview park dr
1,ARBY'S,Arby's,8127 ROUTE 29,8127 lee hwy
2,ARIAKE JAPANESE RESTAURANT,Ariake Japanese Restaurant,8708 LITTLE RIVER TPKE,8708 little river turnpike
4,ARTIE'S,Artie's,3260 BLENHEIM BLVD,3260 old lee hwy
5,AUNTIE ANNE'S,Auntie Anne's,11953U FAIR OAKS SHOPPING CTR,11953 fair oaks mall
7,Anthony's Restaurant,Anthony's Restaurant,3000 ANNANDALE RD,3000 annandale rd
8,BABYLON FUTBOL CAFE,Babylon Futbol Cafe,3501A JEFFERSON ST,3501 s jefferson st
10,BAKERY BREAD & GRILL,"Bakery, Bread & Grill",11186 MAIN ST,11186 lee hwy
11,BAKU DELICIOUS,Baku Delicious,3900 PICKETT RD,3900 pickett rd suite c
12,BAMIAN RESTAURANT,Bamian Restaurant,5634 LEESBURG PIKE,5634 leesburg pike


In [36]:
# Compare street names

# Keep only pairs with matching street numbers
address_matches = possible_matches[
    possible_matches["number_match"]
].copy()

# Remove the street number from Fairfax addresses
address_matches["fairfax_street"] = (
    address_matches["address_clean_fairfax"]
    .str.replace(r"^\d+\s*", "", regex=True)
    .str.strip()
)

# Remove the street number from Google addresses
address_matches["google_street"] = (
    address_matches["street_address"]
    .str.replace(r"^\d+\s*", "", regex=True)
    .str.strip()
)

# Check whether the street names match exactly
address_matches["street_match"] = (
    address_matches["fairfax_street"]
    == address_matches["google_street"]
)

# Count exact matches
print("Matching street names:",
      address_matches["street_match"].sum())

print("Different street names:",
      (~address_matches["street_match"]).sum())

# Preview the matches
display(
    address_matches[
        ["ESTABLISHMENT_NAME", "ADDRESS",
         "street_address", "street_match"]
    ].head(20)
)


Matching street names: 79
Different street names: 90


,ESTABLISHMENT_NAME,ADDRESS,street_address,street_match
0,2941 RESTAURANT,2941 FAIRVIEW PARK DR,2941 fairview park dr,True
1,ARBY'S,8127 ROUTE 29,8127 lee hwy,False
2,ARIAKE JAPANESE RESTAURANT,8708 LITTLE RIVER TPKE,8708 little river turnpike,False
4,ARTIE'S,3260 BLENHEIM BLVD,3260 old lee hwy,False
5,AUNTIE ANNE'S,11953U FAIR OAKS SHOPPING CTR,11953 fair oaks mall,False
7,Anthony's Restaurant,3000 ANNANDALE RD,3000 annandale rd,True
8,BABYLON FUTBOL CAFE,3501A JEFFERSON ST,3501 s jefferson st,False
10,BAKERY BREAD & GRILL,11186 MAIN ST,11186 lee hwy,False
11,BAKU DELICIOUS,3900 PICKETT RD,3900 pickett rd suite c,False
12,BAMIAN RESTAURANT,5634 LEESBURG PIKE,5634 leesburg pike,True


In [37]:
#Improve street address matching

# Make a copy of the pairs with matching street numbers
address_matches = address_matches.copy()

# Remove common suite/unit information and standardize abbreviations
def simplify_street(address):
    if pd.isna(address):
        return ""

    address = str(address).lower()

    # Standardize common abbreviations
    address = re.sub(r"\bturnpike\b", "tpke", address)
    address = re.sub(r"\bhighway\b", "hwy", address)
    address = re.sub(r"\bparkway\b", "pkwy", address)

    # Remove suite or unit information
    address = re.sub(
        r"\b(suite|ste|unit|apt)\s+\w+",
        "",
        address
    )

    # Remove extra spaces
    address = re.sub(r"\s+", " ", address).strip()

    return address

# Apply the same cleaning to both street columns
address_matches["fairfax_simple"] = (
    address_matches["fairfax_street"].apply(simplify_street)
)

address_matches["google_simple"] = (
    address_matches["google_street"].apply(simplify_street)
)

# Compare the simplified addresses
address_matches["street_match_improved"] = (
    address_matches["fairfax_simple"]
    == address_matches["google_simple"]
)

# Count how many matches we now have
print("Matching street names after cleaning:",
      address_matches["street_match_improved"].sum())

print("Still different:",
      (~address_matches["street_match_improved"]).sum())

# Preview the results
display(
    address_matches[
        ["ESTABLISHMENT_NAME", "ADDRESS",
         "street_address", "street_match_improved"]
    ].head(20)
)

Matching street names after cleaning: 85
Still different: 84


,ESTABLISHMENT_NAME,ADDRESS,street_address,street_match_improved
0,2941 RESTAURANT,2941 FAIRVIEW PARK DR,2941 fairview park dr,True
1,ARBY'S,8127 ROUTE 29,8127 lee hwy,False
2,ARIAKE JAPANESE RESTAURANT,8708 LITTLE RIVER TPKE,8708 little river turnpike,True
4,ARTIE'S,3260 BLENHEIM BLVD,3260 old lee hwy,False
5,AUNTIE ANNE'S,11953U FAIR OAKS SHOPPING CTR,11953 fair oaks mall,False
7,Anthony's Restaurant,3000 ANNANDALE RD,3000 annandale rd,True
8,BABYLON FUTBOL CAFE,3501A JEFFERSON ST,3501 s jefferson st,False
10,BAKERY BREAD & GRILL,11186 MAIN ST,11186 lee hwy,False
11,BAKU DELICIOUS,3900 PICKETT RD,3900 pickett rd suite c,True
12,BAMIAN RESTAURANT,5634 LEESBURG PIKE,5634 leesburg pike,True


In [42]:
#Check unique restaurant matches

# Keep pairs with matching street addresses
exact_matches = address_matches[
    address_matches["street_match_improved"]
].copy()

# Count how often each Fairfax restaurant appears
fairfax_counts = exact_matches["RECORDID"].value_counts()

# Count how often each Google restaurant appears
google_counts = exact_matches["gmap_id"].value_counts()

# Check whether each restaurant has only one match
exact_matches["unique_match"] = (
    exact_matches["RECORDID"].map(fairfax_counts).eq(1)
    & exact_matches["gmap_id"].map(google_counts).eq(1)
)

print("Matching address pairs:", len(exact_matches))

print("Unique one-to-one matches:",
      exact_matches["unique_match"].sum())

print("Ambiguous pairs:",
      (~exact_matches["unique_match"]).sum())

# Preview one-to-one matches
display(
    exact_matches.loc[
        exact_matches["unique_match"],
        ["ESTABLISHMENT_NAME", "name",
         "ADDRESS", "street_address", "avg_rating"]
    ].head(15)
)

Matching address pairs: 85
Unique one-to-one matches: 85
Ambiguous pairs: 0


,ESTABLISHMENT_NAME,name,ADDRESS,street_address,avg_rating
0,2941 RESTAURANT,2941 Restaurant,2941 FAIRVIEW PARK DR,2941 fairview park dr,4.6
2,ARIAKE JAPANESE RESTAURANT,Ariake Japanese Restaurant,8708 LITTLE RIVER TPKE,8708 little river turnpike,4.7
7,Anthony's Restaurant,Anthony's Restaurant,3000 ANNANDALE RD,3000 annandale rd,4.4
11,BAKU DELICIOUS,Baku Delicious,3900 PICKETT RD,3900 pickett rd suite c,4.8
12,BAMIAN RESTAURANT,Bamian Restaurant,5634 LEESBURG PIKE,5634 leesburg pike,4.0
15,BARTACO,bartaco,2920 DISTRICT AVE,2920 district ave suite 120,4.6
16,BEBOP KOREAN-MEXICAN GRILL,Bebop Korean-Mexican Grill,10730 FAIRFAX BLVD,10730 fairfax blvd,4.5
17,BELLISSIMO RESTAURANT,Bellissimo Restaurant,10403 MAIN ST,10403 main st,4.4
18,BLUE IGUANA,Blue Iguana,12727 SHOPPES LN,12727 shoppes ln,4.3
20,BUFFALO WILD WINGS,Buffalo Wild Wings,11204 JAMES SWART CIR,11204 james swart cir,3.8


Combining Google Ratings with inspection records

In [43]:
#Combine Google ratings with inspection records

# Keep only unique one-to-one restaurant matches
verified_candidates = exact_matches[
    exact_matches["unique_match"]
].copy()

# Keep the identifiers and Google rating information
restaurant_ratings = verified_candidates[
    ["RECORDID", "gmap_id", "avg_rating",
     "num_of_reviews", "price"]
].copy()

# Connect ratings to all inspection visits
combined_data = pd.merge(
    merged_inspections,
    restaurant_ratings,
    on="RECORDID",
    how="inner",
    validate="many_to_one"
)

# Check our combined dataset
print("Combined dataset shape:", combined_data.shape)

print("Unique restaurants:",
      combined_data["RECORDID"].nunique())

print("Inspection visits:",
      len(combined_data))

# Preview the combined information
display(
    combined_data[
        ["ESTABLISHMENT_NAME", "INSPECTIONDATE",
         "INSPECTIONRESULT", "avg_rating",
         "num_of_reviews"]
    ].head(10)
)

Combined dataset shape: (318, 19)
Unique restaurants: 85
Inspection visits: 318


,ESTABLISHMENT_NAME,INSPECTIONDATE,INSPECTIONRESULT,avg_rating,num_of_reviews
0,2941 RESTAURANT,2025-01-03,Passed,4.6,684
1,2941 RESTAURANT,2025-08-28,Passed,4.6,684
2,2941 RESTAURANT,2026-03-04,Passed,4.6,684
3,2941 RESTAURANT,2026-08-25,Passed,4.6,684
4,ARIAKE JAPANESE RESTAURANT,2025-03-11,Passed,4.7,198
5,ARIAKE JAPANESE RESTAURANT,2025-08-12,Passed,4.7,198
6,ARIAKE JAPANESE RESTAURANT,2026-02-11,Passed,4.7,198
7,ARIAKE JAPANESE RESTAURANT,2026-08-11,Passed,4.7,198
8,Anthony's Restaurant,2025-01-21,Partial Pass,4.4,503
9,Anthony's Restaurant,2025-07-22,Passed,4.4,503


In [45]:
# Check inspection results in our matched sample
print(combined_data["INSPECTIONRESULT"].value_counts())

# Check whether Google ratings have missing values
print("\nMissing Google ratings:",
      combined_data["avg_rating"].isnull().sum())

INSPECTIONRESULT
Passed                  223
Partial Pass             89
Failed - Self Closed      6
Name: count, dtype: int64

Missing Google ratings: 0


Restaurant level dataset

In [46]:
#Keep the most recent inspection for each restaurant

# Make a copy of our combined dataset
restaurant_data = combined_data.copy()

# Convert inspection dates to datetime
restaurant_data["INSPECTIONDATE"] = pd.to_datetime(
    restaurant_data["INSPECTIONDATE"]
)

# Sort from oldest to newest inspection
restaurant_data = restaurant_data.sort_values(
    "INSPECTIONDATE"
)

# Keep the most recent inspection for each restaurant
restaurant_data = restaurant_data.drop_duplicates(
    subset="RECORDID",
    keep="last"
).copy()

# Check our final restaurant-level dataset
print("Final restaurant dataset:", restaurant_data.shape)

print("\nInspection results:")
print(restaurant_data["INSPECTIONRESULT"].value_counts())

# Preview the final dataset
display(
    restaurant_data[
        ["ESTABLISHMENT_NAME", "INSPECTIONDATE",
         "INSPECTIONRESULT", "avg_rating",
         "num_of_reviews"]
    ].head(10)
)

Final restaurant dataset: (85, 19)

Inspection results:
INSPECTIONRESULT
Passed          59
Partial Pass    26
Name: count, dtype: int64


,ESTABLISHMENT_NAME,INSPECTIONDATE,INSPECTIONRESULT,avg_rating,num_of_reviews
290,TROPICAL SMOOTHIE CAFE,2025-11-17,Passed,4.5,187
277,Subway,2026-01-07,Passed,3.4,45
291,TROPICAL SMOOTHIE CAFE,2026-02-03,Passed,4.3,56
302,VIVI BUBBLE TEA,2026-03-06,Partial Pass,4.3,84
46,CALIFORNIA SHABU SHABU,2026-04-06,Passed,4.5,248
215,PEKING VILLAGE II,2026-04-14,Passed,3.4,57
273,Smoothie King,2026-04-27,Partial Pass,4.3,76
203,PANERA BREAD,2026-04-29,Passed,4.2,175
95,ELEPHANT JUMPS,2026-04-30,Partial Pass,4.3,288
114,HAIFA GRILL,2026-05-01,Passed,4.2,607


We are choosing each restaurant's latest recorded inspection, rather than treating multiple visits at the same restaurant as independent observations.

## Explore data

In [38]:
# Summarize and visualize the prepared data.

## Build models

In [39]:
# Train baseline and candidate models.

## Evaluate models

In [40]:
# Compare models using the chosen evaluation measures.

## Final results

In [41]:
# Generate the final tables and figures.

## AI usage disclosure

- **AI tool/model:** [Name the tool and model, or state that no generative AI was used.]
- **Used for:** [Briefly identify the tasks.]
- **How we verified the work:** [Briefly identify the checks.]
- **Prompt(s) used:** [Paste the prompts used.]